In [ ]:
# Set up
%load_ext autoreload
%autoreload 2
%load_ext tensorboard

In [ ]:
import os
import sys

import lightning as L
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
from loguru import logger
from pydantic import BaseModel
from torch.utils.data import DataLoader

import mlflow

load_dotenv()

sys.path.insert(0, "..")

from src.bert4rec.model import BERT4Rec
from src.bert4rec.dataset import BERT4RecDataset, create_bert4rec_datasets
from src.bert4rec.trainer import LitBERT4Rec
from src.id_mapper import IDMapper

import warnings
warnings.filterwarnings("ignore")

# Controller

In [ ]:
max_epochs = 1

In [ ]:
class Args(BaseModel):
    testing: bool = False
    log_to_mlflow: bool = True
    experiment_name: str = "recsys"
    run_name: str = "003-bert4rec"
    notebook_persist_dp: str = None
    random_seed: int = 41
    device: str = None

    batch_size: int = 128

    user_col: str = "user_id"
    item_col: str = "parent_asin"
    rating_col: str = "rating"
    timestamp_col: str = "timestamp"

    top_K: int = 100
    top_k: int = 10

    # BERT4Rec hyperparameters
    max_seq_len: int = 50
    embed_dim: int = 256
    num_layers: int = 6
    num_heads: int = 8
    ff_dim: int = 1024
    dropout_prob: float = 0.1
    mask_prob: float = 0.15

    early_stopping_patience: int = 5
    learning_rate: float = 0.001
    l2_reg: float = 1e-5

    mlf_model_name: str = "bert4rec"

    def init(self):
        self.notebook_persist_dp = os.path.abspath(f"data/{self.run_name}")
        os.makedirs(self.notebook_persist_dp, exist_ok=True)

        if not (mlflow_uri := os.environ.get("MLFLOW_TRACKING_URI")):
            logger.warning(
                f"Environment variable MLFLOW_TRACKING_URI is not set. Setting self.log_to_mlflow to false."
            )
            self.log_to_mlflow = False

        if self.log_to_mlflow:
            logger.info(
                f"Setting up MLflow experiment {self.experiment_name} - run {self.run_name}..."
            )
            self._mlf_logger = MLFlowLogger(
                experiment_name=self.experiment_name,
                run_name=self.run_name,
                tracking_uri=mlflow_uri,
                log_model=True,
            )

        if self.device is None:
            self.device = (
                "cuda"
                if torch.cuda.is_available()
                else "mps" if torch.backends.mps.is_available() else "cpu"
            )

        return self


args = Args().init()

print(args.model_dump_json(indent=2))

# Prep Data

In [ ]:
train_df = pd.read_parquet("../data/train_features_neg_df.parquet")
val_df = pd.read_parquet("../data/val_features_neg_df.parquet")
idm_fp = "../data/idm.json"
idm = IDMapper().load(idm_fp)

assert (
    train_df[args.user_col].map(lambda s: idm.get_user_index(s))
    != train_df["user_indice"]
).sum() == 0, "Mismatch IDM"
assert (
    val_df[args.user_col].map(lambda s: idm.get_user_index(s)) != val_df["user_indice"]
).sum() == 0, "Mismatch IDM"

In [ ]:
# Get real item indices from training data
item_indices = train_df["item_indice"].unique()
n_items = len(item_indices)
logger.info(f"Real n_items: {n_items}")

# Create Datasets

In [ ]:
# Test with mock data (like sequence notebook)
embedding_dim = 8
batch_size = 2

item_sequences = [
    [-1, -1, 2, 3],
    [-1, -1, 2, 3],
    [-1, -1, 1, 3],
    [-1, -1, 2, 1],
    [-1, -1, 2, 1],
]

mock_item_indices = [0, 1, 2, 3, 4]
mock_n_items = len(set(mock_item_indices))  # = 5

mock_train_df = pd.DataFrame({
    "item_sequence": item_sequences,
    "user_indice": [0, 0, 1, 2, 2],
    "item_indice": mock_item_indices,
    "timestamp": [0, 1, 2, 3, 4],
    "rating": [0, 4, 5, 3, 0],
})

# Create mock dataset
mock_dataset = BERT4RecDataset(
    df=mock_train_df,
    max_seq_len=4,
    mask_prob=0.15,
    padding_idx=mock_n_items,
    num_items=mock_n_items,
    for_inference=False
)

# Test dataset directly
sample_idx = 0
sample_data = mock_dataset[sample_idx]

mock_loader = DataLoader(mock_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

# Test model with mock data
test_model = BERT4Rec(
    num_items=mock_n_items,  # = 5
    max_seq_len=4,
    embed_dim=embedding_dim,
    num_layers=2,
    num_heads=2,
    ff_dim=16,
    dropout_prob=0.1,
    padding_idx=mock_n_items  # = 5
)

# Test forward pass
batch = next(iter(mock_loader))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
labels = batch["labels"]

test_model.eval()
with torch.no_grad():
    logits = test_model(input_ids, attention_mask)

    # Test recommendation
    top_k_items, top_k_scores = test_model.recommend(input_ids, k=2, attention_mask=attention_mask)
    logger.info(f"Top-k items shape: {top_k_items.shape}")
    logger.info(f"Top-k scores shape: {top_k_scores.shape}")
    logger.info(f"Top-k items: {top_k_items}")
    logger.info(f"Top-k scores: {top_k_scores}")

logger.info("Mock data test passed!")

In [ ]:
# Create Datasets
n_items = len(item_indices)  # Use real n_items from train data

train_dataset, val_dataset = create_bert4rec_datasets(
    train_df=train_df,
    val_df=val_df,
    max_seq_len=args.max_seq_len,
    mask_prob=args.mask_prob,
    padding_idx=n_items,  # Use n_items as padding (positive index)
    num_items=n_items
)

train_loader = DataLoader(
    train_dataset, batch_size=args.batch_size, shuffle=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=args.batch_size, shuffle=False, drop_last=False
)

logger.info(f"Real n_items: {n_items}")

# Test Model

In [ ]:
# Test with small model
test_model = BERT4Rec(
    num_items=n_items,
    max_seq_len=args.max_seq_len,
    embed_dim=64,
    num_layers=2,
    num_heads=4,
    ff_dim=256,
    dropout_prob=args.dropout_prob,
    padding_idx=n_items
)

# Count parameters
total_params = sum(p.numel() for p in test_model.parameters())
logger.info(f"Test model parameters: {total_params:,.0f}")

In [ ]:
# Test forward pass
batch = next(iter(train_loader))
input_ids = batch["input_ids"][:4]  # Small batch
attention_mask = batch["attention_mask"][:4]

logger.info(f"Input IDs: {input_ids}")
logger.info(f"Input IDs max: {input_ids.max().item()}")
logger.info(f"Input IDs min: {input_ids.min().item()}")
logger.info(f"n_items: {n_items}")

if input_ids.max().item() > n_items:
    logger.error(f"Input IDs contain invalid index > n_items!")
    logger.error(f"Invalid indices: {input_ids[input_ids > n_items]}")

test_model.eval()
with torch.no_grad():
    logits = test_model(input_ids, attention_mask)
    logger.info(f"Logits shape: {logits.shape}")

    # Test recommendation
    top_k_items, top_k_scores = test_model.recommend(input_ids, k=5, attention_mask=attention_mask)
    logger.info(f"Top-k items shape: {top_k_items.shape}")
    logger.info(f"Top-k scores shape: {top_k_scores.shape}")

# Training

In [ ]:
# Initialize model with full hyperparameters
model = BERT4Rec(
    num_items=n_items,
    max_seq_len=args.max_seq_len,
    embed_dim=args.embed_dim,
    num_layers=args.num_layers,
    num_heads=args.num_heads,
    ff_dim=args.ff_dim,
    dropout_prob=args.dropout_prob,
    padding_idx=n_items  # Use n_items as padding (positive index)
)

total_params = sum(p.numel() for p in model.parameters())
logger.info(f"Total parameters: {total_params:,.0f}")

In [ ]:
# Setup callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath=f"{args.notebook_persist_dp}/checkpoints",
    filename="bert4rec-{epoch:02d}-{val_loss:.4f}",
    save_top_k=1,
    monitor="val_loss",
    mode="min",
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=args.early_stopping_patience,
    mode="min",
    verbose=True
)

In [ ]:
# Initialize Lightning trainer
lit_model = LitBERT4Rec(
    model=model,
    learning_rate=args.learning_rate,
    l2_reg=args.l2_reg,
    log_dir=args.notebook_persist_dp,
    evaluate_ranking=True,
    idm=idm,
    args=args,
    checkpoint_callback=checkpoint_callback,
    accelerator=args.device,
    mask_prob=args.mask_prob,
    top_k=args.top_k
)

In [ ]:
# Setup trainer
trainer = L.Trainer(
    default_root_dir=f"{args.notebook_persist_dp}/logs",
    max_epochs=max_epochs,
    accelerator=args.device if args.device else "auto",
    callbacks=[checkpoint_callback, early_stopping],
    logger=args._mlf_logger if args.log_to_mlflow else None,
    log_every_n_steps=10,
    val_check_interval=0.5,  # Validate twice per epoch
    gradient_clip_val=1.0,  # Prevent gradient explosion
    accumulate_grad_batches=1,
)

In [ ]:
# Train model
trainer.fit(
    model=lit_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)

logger.info(f"Training completed. Logs at {trainer.log_dir}")

# Evaluation

In [ ]:
# Load best model
best_model_path = checkpoint_callback.best_model_path
logger.info(f"Loading best model from {best_model_path}")

best_model = LitBERT4Rec.load_from_checkpoint(
    best_model_path,
    model=model
).model

best_model.eval()

In [ ]:
# Test inference
from src.bert4rec.inference import BERT4RecInference

inference_wrapper = BERT4RecInference(
    model=best_model,
    device=args.device,
    max_seq_len=args.max_seq_len,
    padding_idx=n_items  # Use n_items as padding (positive index)
)

# Test with a sample sequence from df
sample_row = val_df.iloc[0]
sample_sequence = sample_row['item_sequence']
recommendations = inference_wrapper.predict_next_item(sample_sequence, top_k=10)

logger.info(f"Sample sequence: {sample_sequence[:10]}...")
logger.info(f"Top-10 recommendations: {recommendations['predicted_items']}")
logger.info(f"Top-10 scores: {recommendations['predicted_scores']}")

# Save to MLflow

In [ ]:
if args.log_to_mlflow:
    logger.info("Saving model to MLflow...")

    # Log model parameters
    mlflow.log_params({
        "num_items": n_items,
        "max_seq_len": args.max_seq_len,
        "embed_dim": args.embed_dim,
        "num_layers": args.num_layers,
        "num_heads": args.num_heads,
        "ff_dim": args.ff_dim,
        "dropout_prob": args.dropout_prob,
        "mask_prob": args.mask_prob,
        "learning_rate": args.learning_rate,
        "batch_size": args.batch_size,
        "total_params": total_params
    })

    # Log model artifact
    mlflow.pytorch.log_model(
        pytorch_model=best_model,
        artifact_path="model",
        registered_model_name=args.mlf_model_name
    )

    logger.info("Model saved to MLflow successfully")

# Tensorboard

In [ ]:
%tensorboard --logdir $trainer.log_dir